## Создаём парсер, который извлекает из XML следующие характеристики каждого слова:

original — исходное написание слова, берём из атрибута original тега word. Если атрибут пустой, оставляем пустую строку;

nucleus / phrasal_stress — фразовое ударение слова. Ставится на ядро синтагмы, если указано тегом nucleus="2". Если ядро не указано, ударение ставится на последнее слово синтагмы;

pause_len — длительность паузы после слова, значение из тега pause. Если пауза меньше 150 мс, считаем её отсутствующей (-1), если больше 800 мс, ограничиваем 800 мс;

phrasal_stress — слово является ядром синтагмы (фразовое ударение);

is_capital — слово с заглавной буквы;

length — длина слова;

fonetic_words_before / fonetic_words_after — количество слов с фонетическим выделением до и после текущего слова в предложении;

word_num — индекс слова в предложении;

sentence_len — количество слов в предложении;

prev_word / next_word — предыдущие и следующие слова (после нормализации);

sintagma_num — номер синтагмы с учетом пауз;

has_pause / pause — наличие паузы и её длительность;

genesys - род и одушевленность слова

In [17]:
import os

# Текст Balzak_daughter

In [51]:
import xml.etree.ElementTree as ET
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

INPUT_XML = "balzak_daughter.Result.xml" # Входной XML файл с разметкой речи
OUTPUT_JSON = "output_Balzak.json" # Выходной JSON с минимальными полями

MIN_PAUSE_MS = 150 # Минимальная длительность паузы
MAX_PAUSE_MS = 800 # Максимальная длительность паузы
PHRASE_END_PUNCT = {".", ",", ":", ";", "!", "?", "…"} # Пунктуация конца фразы

# Функция для определения, является ли слово фонетически выделенным
def is_phonetic(word_elem):
    return word_elem is not None and word_elem.get("nucleus") is not None

def parse_sentence(sentence):
    words_result = []
    current_sintagma = []
    sintagma_num = 0

    for elem in sentence.iter():
        tag = elem.tag.lower()

        # Обработка слов
        if tag == "word":
            text = elem.get("original") or "" # Исходное написание
            nucleus = elem.get("nucleus") # Ядро синтагмы (для ударения)
            genesys = elem.get("genesys") or "" # Род и одушевленность слова
            current_sintagma.append({
                "content": text,
                "nucleus": nucleus,
                "pause_len": -1,
                "phrasal_stress": False,
                "genesys": genesys
            })

        # Обработка пауз
        elif tag == "pause":
            t = elem.get("time")
            pause_ms = -1
            if t:
                try:
                    pause_ms = int(float(t))
                    if pause_ms < MIN_PAUSE_MS:
                        pause_ms = -1
                    elif pause_ms > MAX_PAUSE_MS:
                        pause_ms = MAX_PAUSE_MS
                except:
                    pause_ms = -1
            else:
                # Стандартные значения для типа паузы
                p_type = elem.get("type")
                if p_type == "weak":
                    pause_ms = 300
                elif p_type == "long":
                    pause_ms = 500

            # Присваиваем паузу последнему слову синтагмы
            if current_sintagma:
                current_sintagma[-1]["pause_len"] = pause_ms
                sintagma_num += 1

                # Определяем фразовое ударение
                nucleus_words = [w for w in current_sintagma if w["nucleus"]=="2"]
                if nucleus_words:
                    nucleus_words[0]["phrasal_stress"] = True
                else:
                    current_sintagma[-1]["phrasal_stress"] = True

                for w in current_sintagma:
                    w["sintagma_num"] = sintagma_num
                    words_result.append(w)

                current_sintagma = []

    # Обработка остатка слов без паузы
    if current_sintagma:
        sintagma_num += 1
        current_sintagma[-1]["phrasal_stress"] = True
        for w in current_sintagma:
            w["sintagma_num"] = sintagma_num
            words_result.append(w)

    # Вычисляем дополнительные признаки для модели
    sentence_len = len(words_result)
    for i, w in enumerate(words_result):
        w["is_capital"] = w["content"][:1].isupper() if w["content"] else False
        w["length"] = len(w["content"])
        w["word_num"] = i + 1
        w["sentence_len"] = sentence_len
        w["prev_word"] = words_result[i-1]["content"].lower() if i > 0 else ""
        w["next_word"] = words_result[i+1]["content"].lower() if i < sentence_len-1 else ""
        w["fonetic_words_before"] = sum(1 for j in range(i) if is_phonetic(words_result[j]))
        w["fonetic_words_after"] = sum(1 for j in range(i+1, sentence_len) if is_phonetic(words_result[j]))
        w["has_pause"] = w["pause_len"] > 0

    return words_result

def parse_xml_for_json(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    outputs = []

    for sentence in root.iter("sentence"):
        words = parse_sentence(sentence)
        if words:
            outputs.append({"words": words})
    return outputs

# Парсинг XML
outputs = parse_xml_for_json(INPUT_XML)

# Создаем минимальный JSON
json_result = []
for sentence in outputs:
    words_list = []
    for w in sentence["words"]:
        words_list.append({
            "content": w["content"], # Сохраняем только текст
            "phrasal_stress": w["phrasal_stress"], # Фразовое ударение
            "pause_len": w["pause_len"] # Длительность паузы
        })
    json_result.append({"words": words_list})

# Сохраняем JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_result, f, ensure_ascii=False, indent=4)

print(f"JSON сохранён: {OUTPUT_JSON}")

# Подготовка признаков для модели
rows = [w for sentence in outputs for w in sentence["words"]]
X = [
    [
        len(w["content"]),
        w["content"][:1].isupper() if w["content"] else False,
        w["pause_len"],
        w["is_capital"],
        w["length"],
        w["word_num"],
        w["sentence_len"],
        w["sintagma_num"],
        w["fonetic_words_before"],
        w["fonetic_words_after"]
    ]
    for w in rows
]
y = [w["phrasal_stress"] for w in rows]

# Обучение модели Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred, digits=3))

JSON сохранён: output_Balzak.json
Classification Report
              precision    recall  f1-score   support

       False      0.966     0.987     0.977      5434
        True      0.952     0.881     0.915      1570

    accuracy                          0.963      7004
   macro avg      0.959     0.934     0.946      7004
weighted avg      0.963     0.963     0.963      7004



# Тестовый текст

In [50]:
import xml.etree.ElementTree as ET
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

INPUT_XML = "Test_Procody.xml" # Входной XML файл с разметкой речи
OUTPUT_JSON = "output.json" # Выходной JSON с минимальными полями

MIN_PAUSE_MS = 150 # Минимальная длительность паузы
MAX_PAUSE_MS = 800 # Максимальная длительность паузы
PHRASE_END_PUNCT = {".", ",", ":", ";", "!", "?", "…"} # Пунктуация конца фразы

# Функция для определения, является ли слово фонетически выделенным
def is_phonetic(word_elem):
    return word_elem is not None and word_elem.get("nucleus") is not None

def parse_sentence(sentence):
    words_result = []
    current_sintagma = []
    sintagma_num = 0

    for elem in sentence.iter():
        tag = elem.tag.lower()

        # Обработка слов
        if tag == "word":
            text = elem.get("original") or "" # Исходное написание
            nucleus = elem.get("nucleus") # Ядро синтагмы (для ударения)
            genesys = elem.get("genesys") or "" # Род и одушевленность слова
            current_sintagma.append({
                "content": text,
                "nucleus": nucleus,
                "pause_len": -1,
                "phrasal_stress": False,
                "genesys": genesys
            })

        # Обработка пауз
        elif tag == "pause":
            t = elem.get("time")
            pause_ms = -1
            if t:
                try:
                    pause_ms = int(float(t))
                    if pause_ms < MIN_PAUSE_MS:
                        pause_ms = -1
                    elif pause_ms > MAX_PAUSE_MS:
                        pause_ms = MAX_PAUSE_MS
                except:
                    pause_ms = -1
            else:
                # Стандартные значения для типа паузы
                p_type = elem.get("type")
                if p_type == "weak":
                    pause_ms = 300
                elif p_type == "long":
                    pause_ms = 500

            # Присваиваем паузу последнему слову синтагмы
            if current_sintagma:
                current_sintagma[-1]["pause_len"] = pause_ms
                sintagma_num += 1

                # Определяем фразовое ударение
                nucleus_words = [w for w in current_sintagma if w["nucleus"]=="2"]
                if nucleus_words:
                    nucleus_words[0]["phrasal_stress"] = True
                else:
                    current_sintagma[-1]["phrasal_stress"] = True

                for w in current_sintagma:
                    w["sintagma_num"] = sintagma_num
                    words_result.append(w)

                current_sintagma = []

    # Обработка остатка слов без паузы
    if current_sintagma:
        sintagma_num += 1
        current_sintagma[-1]["phrasal_stress"] = True
        for w in current_sintagma:
            w["sintagma_num"] = sintagma_num
            words_result.append(w)

    # Вычисляем дополнительные признаки для модели
    sentence_len = len(words_result)
    for i, w in enumerate(words_result):
        w["is_capital"] = w["content"][:1].isupper() if w["content"] else False
        w["length"] = len(w["content"])
        w["word_num"] = i + 1
        w["sentence_len"] = sentence_len
        w["prev_word"] = words_result[i-1]["content"].lower() if i > 0 else ""
        w["next_word"] = words_result[i+1]["content"].lower() if i < sentence_len-1 else ""
        w["fonetic_words_before"] = sum(1 for j in range(i) if is_phonetic(words_result[j]))
        w["fonetic_words_after"] = sum(1 for j in range(i+1, sentence_len) if is_phonetic(words_result[j]))
        w["has_pause"] = w["pause_len"] > 0

    return words_result

def parse_xml_for_json(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    outputs = []

    for sentence in root.iter("sentence"):
        words = parse_sentence(sentence)
        if words:
            outputs.append({"words": words})
    return outputs

# Парсинг XML
outputs = parse_xml_for_json(INPUT_XML)

# Создаем минимальный JSON
json_result = []
for sentence in outputs:
    words_list = []
    for w in sentence["words"]:
        words_list.append({
            "content": w["content"], # Сохраняем только текст
            "phrasal_stress": w["phrasal_stress"], # Фразовое ударение
            "pause_len": w["pause_len"] # Длительность паузы
        })
    json_result.append({"words": words_list})

# Сохраняем JSON
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_result, f, ensure_ascii=False, indent=4)

print(f"JSON сохранён: {OUTPUT_JSON}")

# Подготовка признаков для модели
rows = [w for sentence in outputs for w in sentence["words"]]
X = [
    [
        len(w["content"]),
        w["content"][:1].isupper() if w["content"] else False,
        w["pause_len"],
        w["is_capital"],
        w["length"],
        w["word_num"],
        w["sentence_len"],
        w["sintagma_num"],
        w["fonetic_words_before"],
        w["fonetic_words_after"]
    ]
    for w in rows
]
y = [w["phrasal_stress"] for w in rows]

# Обучение модели Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred, digits=3))

JSON сохранён: output.json
Classification Report
              precision    recall  f1-score   support

       False      0.998     1.000     0.999       473
        True      1.000     0.992     0.996       130

    accuracy                          0.998       603
   macro avg      0.999     0.996     0.998       603
weighted avg      0.998     0.998     0.998       603

